<a href="https://colab.research.google.com/github/palaciosruth52-alt/Deteccion_Hojas_Cafe/blob/main/Deteccion_de_Enfermedades_cafe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Deteccion de enfermedades en Café

###Paso #1 Importar Librerias

In [ ]:
import os
import zipfile
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import matplotlib.pyplot as plt

print("Versión de TensorFlow:", tf.__version__)

Versión de TensorFlow: 2.20.0


###Cargar Dataset

####Descomprimir datase.zip

In [4]:
archivo_zip = 'Dataset.zip'
carpeta_destino = 'dataset_cafe'

if os.path.exists(archivo_zip):
    with zipfile.ZipFile(archivo_zip, 'r') as zip_ref:
        zip_ref.extractall(carpeta_destino)
    print("¡Dataset descomprimido exitosamente!")
    print("Contenido:", os.listdir(carpeta_destino))
else:
    print(f"Error: No se encontró el archivo {archivo_zip}.")

¡Dataset descomprimido exitosamente!
Contenido: ['Dataset']


In [ ]:
import os
import zipfile

ruta_dataset = 'dataset_cafe/Dataset'
carpeta_final = 'dataset_final_imagenes'
os.makedirs(carpeta_final, exist_ok=True)

# Buscar y descomprimir cada archivo .zip que está adentro
for archivo in os.listdir(ruta_dataset):
    if archivo.endswith('.zip'):
        ruta_zip = os.path.join(ruta_dataset, archivo)
        try:
            with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:
                zip_ref.extractall(carpeta_final)
            print(f"Descomprimido: {archivo}")
        except Exception as e:
            print(f"No se pudo descomprimir {archivo}: {e}")

print("¡Proceso de descompresión interna terminado!")

No se pudo descomprimir coffee___rust4.zip: File is not a zip file
No se pudo descomprimir red_spider_v2.zip: File is not a zip file
No se pudo descomprimir coffee___healthy.zip: [Errno 21] Is a directory: 'dataset_cafe/Dataset/coffee___healthy.zip'
Descomprimido: coffee__leaf_miner.zip
No se pudo descomprimir cercospora_v2.0_fotoEstudio.zip: File is not a zip file
Descomprimido: coffee__phoma.zip
Descomprimido: coffee___rust.zip
No se pudo descomprimir Miner_Prueba.zip: File is not a zip file
No se pudo descomprimir Phoma_Prueba.zip: File is not a zip file
¡Proceso de descompresión interna terminado!


###Generadores de Datos

In [5]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
ruta_datos = 'dataset_cafe/Dataset'

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_generator = datagen.flow_from_directory(
    ruta_datos, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training'
)

val_generator = datagen.flow_from_directory(
    ruta_datos, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation'
)

print("Clases detectadas:", train_generator.class_indices)

Found 1440 images belonging to 4 classes.
Found 360 images belonging to 4 classes.
Clases detectadas: {'coffee___healthy.zip': 0, 'coffee___rust': 1, 'coffee__leaf_miner': 2, 'coffee__phoma': 3}


In [6]:
import os
print("Contenido dentro de 'dataset_cafe/Dataset':", os.listdir('dataset_cafe/Dataset'))

Contenido dentro de 'dataset_cafe/Dataset': ['coffee__leaf_miner', 'coffee___healthy.zip.001', 'coffee___rust4.zip', 'red_spider_v2.zip', 'coffee___healthy.zip', 'coffee__leaf_miner.zip', 'coffee___healthy.zip.003', 'coffee___rust', 'cercospora_v2.0_fotoEstudio.zip', 'coffee__phoma.zip', 'coffee__phoma', 'coffee___rust.zip', 'coffee___healthy.zip.002', 'Miner_Prueba.zip', 'Phoma_Prueba.zip']


###Crear Modelo

In [7]:
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
num_classes = len(train_generator.class_indices)
output = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,468 (9.24 MB)

 Trainable params: 164,484 (642.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [8]:
EPOCHS = 10

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS
)

Epoch 1/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 125s 3s/step - accuracy: 0.7750 - loss: 0.5039 - val_accuracy: 0.9000 - val_loss: 0.2313
Epoch 2/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 112s 2s/step - accuracy: 0.8667 - loss: 0.2884 - val_accuracy: 0.9139 - val_loss: 0.1947
Epoch 3/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 113s 3s/step - accuracy: 0.8819 - loss: 0.2543 - val_accuracy: 0.9583 - val_loss: 0.1520
Epoch 4/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 113s 3s/step - accuracy: 0.9014 - loss: 0.2224 - val_accuracy: 0.9000 - val_loss: 0.2432
Epoch 5/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 112s 2s/step - accuracy: 0.8833 - loss: 0.2675 - val_accuracy: 0.9444 - val_loss: 0.1355
Epoch 6/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 121s 3s/step - accuracy: 0.9042 - loss: 0.2134 - val_accuracy: 0.9361 - val_loss: 0.1716
Epoch 7/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 114s 3s/step - accuracy: 0.9125 - loss: 0.1917 - val_accuracy: 0.9417 - val_loss: 0.1300
Epoch 8/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 113s 2s/step - accuracy: 0.9229 - loss: 0.1835 - val_accuracy: 0.9389 - v

###Guardar y descargar

In [9]:
nombre_modelo = 'modelo_hojas_cafe.h5'
model.save(nombre_modelo)
print(f"¡Modelo guardado como '{nombre_modelo}'!")

from google.colab import files
files.download(nombre_modelo)

¡Modelo guardado como 'modelo_hojas_cafe.h5'!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>